In [1]:
from verify_objects import *
from verify_relation import *
from verifier import *

# Example2: Accelerated Gradient Descent

## 1. Positiveness of $\theta_k$

### Proving the alternative form of $\theta_k$ satisfies the recurrence relation

In [2]:
t = ScalarSequence('t')

def t_rel(n):
    return Equal( t(n+1)*t(n+1) - t(n+1), t(n)*t(n) )

#### Defining the alternative form

In [3]:
assumption_t0 = Equal(t(0),1)
def t_def(n):
    return Equal(t(n+1), 1/2*(Sqrt(4*t(n)*t(n)+1) +1))

Prop.add( assumption_t0 )
Prop.add( t_def )

propadd_assumption(t_def)
propadd_intro(t_def,arb_n)

#### Verifying the equivalence

In [4]:
verify(t_rel, [t_def(arb_n)])

True

## Verifying the positiveness of $\theta_k$

###  Declaring our goal

In [5]:
def t_pos(n): return lt(0,t(n))

#### Definition of $\theta_k$

In [6]:
Prop.clear()

# Assume t_0 = 1 and the recurrence relation
propadd_assumption(assumption_t0) 
propadd_assumption(t_def)    

# Specifically we consider t(0), t(arb_n) to be true
propadd_intro(t_def, 0)       
propadd_intro(t_def, arb_n)

In [7]:
propadd_assumption(sqrt_nonneg) # sqrt is nonneg
propadd_intro(sqrt_nonneg, Sqrt(4*t(arb_n)*t(arb_n)+1)) # 0<= Sqrt(4*t(arb_n)*t(arb_n)+1)

step_a = add_of(le(0,Sqrt(4*t(arb_n)*t(arb_n)+1)),1) # 1 <=  Sqrt(4*t(arb_n)*t(arb_n)+1) + 1

In [8]:
propadd_ineq(lt(0,1),[]) # 0 < 1

step_b = ineq_trans(lt(0,1),le(1, Sqrt(4*t(arb_n)*t(arb_n)+1)+1)) #0 <  Sqrt(4*t(arb_n)*t(arb_n)+1) + 1  by transitivity

In [9]:
propadd_ineq(lt(0,2),[]) # 0 < 2
step_c = divide_inequality( lt(0, Sqrt(4*t(arb_n)*t(arb_n)+1)+1 ) , 2) # dividing by positive number preserves inequality, and 

propadd_simplify( step_c ) #temp

In [10]:
induction( t_pos, [[assumption_t0]] , [[t_def(arb_n).simplify()]] )

case 0 : 0 < t_{0} 
case succ : 0 < t_{1+arb_n} 


## 2. Equivalence of two method expressions

## x,y equiv

In [11]:
#Basic Setting
Prop.clear()

x = Sequence('x')
y = Sequence('y')

X = Sequence('X') #For equivalent formulation
Y = Sequence('Y')
Z = Sequence('Z')

f = Function('f')
g = Operator('g') # Grdaient f
k = IterationCount('k')
L = Scalar('L')   # L-smoothness

In [12]:
#Define theta
t = ScalarSequence('t')

assumption_t0 = Equal(t(0),1)
def t_def(n):
    return Equal(t(n+1), 1/2*(Sqrt(4*t(n)*t(n)+1) +1))

Prop.add(assumption_t0)
Prop.add(t_def)

In [13]:
#Fomulation 1

assumption1 = Equal(t(0),1)
assumption2 = Equal(y(0),x(0))

def form1eq1(n):
    return Equal(y(n+1), x(n)-1/L*g(x(n)))

def form1eq2(n):
    return Equal(x(n+1) , y(n+1) + (t(n)-1)/t(n+1)*(y(n+1)-y(n)))

Prop.update([t_def,assumption1,assumption2,form1eq1,form1eq2])


#Formulation 2
assumption3 = Equal(Y(0) , X(0))
assumption4 = Equal(Z(0) , X(0))
assumption5 = Equal(x(0) , X(0))
assumption6 = Equal(y(0) , Y(0))


def form2eq1(n):
    return Equal(Y(n+1), X(n) - 1/L*g(X(n)))

def form2eq2(n):
    return Equal(Z(n+1), Z(n) - t(n)/L*g(x(n)))

def form2eq3(n):
    return Equal(X(n+1), (1- 1/t(n+1))*Y(n+1) + 1/t(n+1)*Z(n+1) )


Prop.update([assumption3, assumption4, assumption5, assumption6, form2eq1, form2eq2, form2eq3])
tmp = Prop.copy()

In [14]:
#Our goals

def x_equiv(n):
    return Equal(x(n),X(n))

def y_equiv(n):
    return Equal(y(n),Y(n))

xy_equiv = prop_and([x_equiv,y_equiv])

y-equivalence (under x assumptions)

In [15]:
Prop.add(Equal(x(arb_n),X(arb_n)))
propadd_intro(form1eq1,arb_n)
propadd_intro(form2eq1,arb_n)
induction(y_equiv,[[]],[[form1eq1(arb_n),form2eq1(arb_n),Equal(x(arb_n),X(arb_n))]])

case 0 : True 
case succ : True 


x_equivalence (under x_assumptions)

In [16]:
# induction hypothersis
Prop.add( y_equiv(arb_n) )
Prop.add( y_equiv(arb_n+1) )
Prop.add( x_equiv(arb_n) )

propadd_intro( form1eq2, arb_n )
propadd_intro( form2eq2, arb_n )
propadd_intro( form2eq3, arb_n-1 )
propadd_intro( form2eq3, arb_n )

def_z_n = Equal(Z(arb_n), t(arb_n)*X(arb_n) + (1-t(arb_n))*Y(arb_n))
propadd_eq(def_z_n, [form2eq3(arb_n-1) ])

In [17]:
def stepp(result,step):
    if step in Prop:
        print(f'before: {result}\n step: {step}')
        tmp_res = result
        if isinstance(step, Equal):                 #substitution
            try: result = result.substitute(step)
            except: pass
            print(f'after : {result}\n  changed : {not tmp_res == result} \n ')
        elif isinstance(step, lemma):               #add lemma
              propadd_eq(lemma.claim, lemma.proof)
        else: raise TypeError(f'Cannot use the following step: {step}' )
    else:
        raise TypeError( f'Should first verify the current step: {step}')

    try:
        print(f'simplified to : {result.simplify()}\n')
        return result.simplify()
    except:
        print('not simplified \n')
        return result

In [18]:
s = [0] * 100
s[0] = X(arb_n+1) - x(arb_n+1)

proofff = [form2eq3(arb_n),form1eq2(arb_n), y_equiv(arb_n+1),y_equiv(arb_n), form2eq2(arb_n), def_z_n,form2eq1(arb_n),x_equiv(arb_n)]
for i in range(len(proofff)):
     s[i+1] = stepp(s[i],proofff[i])

verify( x_equiv(arb_n+1), proofff )

before: -x^{1+arb_n} + X^{1+arb_n}
 step: X^{1+arb_n} = ( 1-t_{1+arb_n}**{-1} ) * Y^{1+arb_n} + t_{1+arb_n}**{-1} * Z^{1+arb_n}
after : ( 1-t_{1+arb_n}**{-1} ) * Y^{1+arb_n}-x^{1+arb_n} + t_{1+arb_n}**{-1} * Z^{1+arb_n}
  changed : True 
 
simplified to : ( 1-t_{1+arb_n}**{-1} ) * Y^{1+arb_n}-x^{1+arb_n} + t_{1+arb_n}**{-1} * Z^{1+arb_n}

before: ( 1-t_{1+arb_n}**{-1} ) * Y^{1+arb_n}-x^{1+arb_n} + t_{1+arb_n}**{-1} * Z^{1+arb_n}
 step: x^{1+arb_n} = ( 1+( -1+t_{arb_n} ) * t_{1+arb_n}**{-1} ) * y^{1+arb_n}-( -1+t_{arb_n} ) * t_{1+arb_n}**{-1} * y^{arb_n}
after : ( -1+t_{arb_n} ) * t_{1+arb_n}**{-1} * y^{arb_n} + ( 1-t_{1+arb_n}**{-1} ) * Y^{1+arb_n}-( 1+( -1+t_{arb_n} ) * t_{1+arb_n}**{-1} ) * y^{1+arb_n} + t_{1+arb_n}**{-1} * Z^{1+arb_n}
  changed : True 
 
simplified to : ( -1-t_{1+arb_n}**{-1} * t_{arb_n}+t_{1+arb_n}**{-1} ) * y^{1+arb_n} + ( -t_{1+arb_n}**{-1}+t_{1+arb_n}**{-1} * t_{arb_n} ) * y^{arb_n} + ( 1-t_{1+arb_n}**{-1} ) * Y^{1+arb_n} + t_{1+arb_n}**{-1} * Z^{1+arb_n}

befor

True

## Combine together to complete induction process

In [19]:
Prop = tmp.copy()
propadd_intro(form1eq1, arb_n)
propadd_intro(form1eq2, arb_n)
propadd_intro(form2eq1, arb_n)
propadd_intro(form2eq2, arb_n)
propadd_intro(form2eq3, arb_n-1)
propadd_intro(form2eq3, arb_n)

def_z_n = Equal( Z(arb_n), t(arb_n) * X(arb_n) + (1-t(arb_n)) * Y(arb_n) )
propadd_eq( def_z_n, [ form2eq3(arb_n-1) ] )

In [20]:
proof_y = [ form1eq1(arb_n), form2eq1(arb_n), Equal(x(arb_n), X(arb_n)) ]
lem_y_equiv = lemma( y_equiv(arb_n+1), proof_y )
proof_x = [ lem_y_equiv, form2eq3(arb_n), form1eq2(arb_n), y_equiv(arb_n+1), y_equiv(arb_n), form2eq2(arb_n), def_z_n, form2eq1(arb_n), x_equiv(arb_n) ]

induction( xy_equiv, [[],[]], [proof_x, proof_y] )

case 0 : True 
case succ : True 


---
## Lyapunov analysis
$$ $$
---

## Define materials

In [21]:
x = Sequence('x') # seqeunce generated by gradient descent
y = Sequence('y')
z = Sequence('z')
f = Function('f')
g = Operator('g') # Grdaient f
k = IterationCount('k')
L = Scalar('L')   # smoothness parameter
A = ScalarSequence('A')
t = ScalarSequence('t')

In [22]:
x_star = x('\star')
f_star = f(x_star)

### Define Lyapunov function
$$ \Phi(n) = A_n ( f(x^n) - f^\star ) + \frac{L}{2} \left\| z^n - x^\star \right\|^2 $$

In [23]:
def Lypunov(n):
    return A(n) * ( f(x(n)) - f_star )+ L/2 * NS( z(n)-x_star )

## The 'proof' we want to show is
$$
\begin{aligned}
\Phi(k+1)-\Phi(k)
&= \lambda_1 \left( f(y^k)-f^\star+\left\langle \nabla f(x^k),x^\star-x^k \right\rangle \right) \\
&\quad + \lambda_2 \left( f(y^k)-f(x^k)+\left\langle \nabla f(y^k),x^k-y^k \right\rangle \right) \\
&\quad + \lambda_3 \left( f(x^{k+1})-f(y^k)+\left\langle \nabla f(y^k),x^{k+1}-y^k \right\rangle+\frac{L}{2}\|y^k-x^{k+1}\|^2 \right) \\
&\quad - \lambda_4 \frac{1}{2L}\|\nabla f(x^k)\|^2
\end{aligned}
$$
### where
$$
\begin{aligned}
\lambda_1 &= A_{k+1}-A_k, \\
\lambda_2 &= A_k, \\
\lambda_3 &= A_{k+1}, \\
\lambda_4 &= \frac{A_{k+1}-(A_k-A_{k+1})^2}{2L}.
\end{aligned}
$$

In [24]:
lambda_1 = A(k+1) - A(k)
lambda_2 = A(k)
lambda_3 = A(k+1)
lambda_4 = (A(k+1)-(A(k)-A(k+1))**2)/(2*L)

In [25]:
ineq1 = f(y(k)) - f_star + IP(g(y(k)), x_star - y(k))
ineq2 = f(y(k)) - f(x(k)) + IP(g(y(k)),x(k)-y(k))
ineq3 = f(x(k+1)) - f(y(k)) - IP(g(y(k)), x(k+1) - y(k)) - L/2 * NS( x(k+1) - y(k) )
ineq4 = -NS(g(y(k)))

In [26]:
LHS = Lypunov(k+1) - Lypunov(k)
RHS = lambda_1 * ineq1 + lambda_2 * ineq2 + lambda_3 * ineq3 + lambda_4 * ineq4

In [27]:
LHS - RHS 

( - f(x^{\star})+f(x^{1+k}) ) A_{1+k}-( - < -y^{k} + x^{1+k}, g(y^{k}) >- f(y^{k})-0.5 L || -y^{k} + x^{1+k} ||^2+f(x^{1+k}) ) A_{1+k}-( - f(x^{\star})+< -y^{k} + x^{\star}, g(y^{k}) >+f(y^{k}) ) ( -A_{k}+A_{1+k} )-( - f(x^{\star})+f(x^{k}) ) A_{k}-( - f(x^{k})+< -y^{k} + x^{k}, g(y^{k}) >+f(y^{k}) ) A_{k}-0.5 L || -x^{\star} + z^{k} ||^2+0.5 ( -(-A_{1+k}+A_{k})^{2}+A_{1+k} ) L^{-1} || g(y^{k}) ||^2+0.5 L || -x^{\star} + z^{1+k} ||^2

In [28]:
( LHS - RHS ).simplify()

( -0.5 A_{1+k}^{2} L^{-1}-0.5 A_{k}^{2} L^{-1}+0.5 A_{1+k} L^{-1}+A_{1+k} A_{k} L^{-1} ) || g(y^{k}) ||^2+( -A_{1+k}+A_{k} ) < g(y^{k}), x^{\star} >-0.5 L || z^{k} ||^2-A_{1+k} L < x^{1+k}, y^{k} >-A_{k} < g(y^{k}), x^{k} >-L < x^{\star}, z^{1+k} >+0.5 A_{1+k} L || x^{1+k} ||^2+0.5 A_{1+k} L || y^{k} ||^2+0.5 L || z^{1+k} ||^2+A_{1+k} < g(y^{k}), x^{1+k} >+L < x^{\star}, z^{k} >

## Substituting definitions and simplifying...

In [29]:
def z_def(n) :
    statement = Equal(z(n+1), z(n) - (A(n+1) - A(n))/L * g( y(n) ) )
    statement.truth = 'Axiom'
    return statement

def x_def(n) :
    statement = Equal(x(n+1), y(n) - 1/L * g( y(n) ) )
    statement.truth = 'Axiom'
    return statement

def A_def(n):
    statement = Equal(A(n+1), A(n)+t(n))
    statement.truth = 'Axiom'
    return statement

In [30]:
def x_rel(n) :
    statement = Equal(x(n), A(n+1)/A(n) * y(n) - ( A(n+1)/A(n) - 1 ) * z(n) )
    statement.truth = 'Axiom'
    return statement

In [31]:
( LHS - RHS ).substitute(z_def(k)).substitute(x_def(k)).substitute(x_rel(k)).substitute(A_def(k)).simplify()

0